# 02 City Reference Model

## Purpose
Create the stable city reference table used by later source joins.

## Inputs
Local city constants for eight controlled starter cities.

## Outputs
`data/silver/city_reference.csv` and `data/silver/city_reference.parquet` when explicitly written.

## Technologies used
Python, pandas, pyarrow, Jupyter.

## Configuration
Uses project-relative `data/silver`. No external calls.

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd()
DATA_DIR = Path(os.getenv("DATA_DIR", "data"))
CHECKPOINT_DIR = Path(os.getenv("CHECKPOINT_DIR", "data/checkpoints"))


## Implementation
The city reference logic is migrated from the legacy `src/city_mapping/build_city_reference.py` module. It creates stable `city_id` values and validates join keys.

In [ ]:
from pathlib import Path
import pandas as pd

CITY_RECORDS = [
    {"city_id": "vienna_at", "city_name": "Vienna", "city_name_normalized": "vienna", "country_code": "AT", "latitude": 48.2082, "longitude": 16.3738, "population": None, "area_km2": None, "population_density": None, "mapping_notes": "Phase 1 pilot; all sources feasible with constraints.", "eea_station_selection_notes": "Use selected Vienna EEA station after file verification.", "wikipedia_url": "https://en.wikipedia.org/wiki/Vienna"},
    {"city_id": "berlin_de", "city_name": "Berlin", "city_name_normalized": "berlin", "country_code": "DE", "latitude": 52.5200, "longitude": 13.4050, "population": None, "area_km2": None, "population_density": None, "mapping_notes": "Phase 1 pilot; all sources feasible with constraints.", "eea_station_selection_notes": "Use selected Berlin EEA station after file verification.", "wikipedia_url": "https://en.wikipedia.org/wiki/Berlin"},
    {"city_id": "paris_fr", "city_name": "Paris", "city_name_normalized": "paris", "country_code": "FR", "latitude": 48.8566, "longitude": 2.3522, "population": None, "area_km2": None, "population_density": None, "mapping_notes": "Starter city for major European capital coverage.", "eea_station_selection_notes": "Use selected Paris EEA station after file verification.", "wikipedia_url": "https://en.wikipedia.org/wiki/Paris"},
    {"city_id": "madrid_es", "city_name": "Madrid", "city_name_normalized": "madrid", "country_code": "ES", "latitude": 40.4168, "longitude": -3.7038, "population": None, "area_km2": None, "population_density": None, "mapping_notes": "Starter city for southern European comparison.", "eea_station_selection_notes": "Use selected Madrid EEA station after file verification.", "wikipedia_url": "https://en.wikipedia.org/wiki/Madrid"},
    {"city_id": "rome_it", "city_name": "Rome", "city_name_normalized": "rome", "country_code": "IT", "latitude": 41.9028, "longitude": 12.4964, "population": None, "area_km2": None, "population_density": None, "mapping_notes": "Starter city for Mediterranean comparison.", "eea_station_selection_notes": "Use selected Rome EEA station after file verification.", "wikipedia_url": "https://en.wikipedia.org/wiki/Rome"},
    {"city_id": "amsterdam_nl", "city_name": "Amsterdam", "city_name_normalized": "amsterdam", "country_code": "NL", "latitude": 52.3676, "longitude": 4.9041, "population": None, "area_km2": None, "population_density": None, "mapping_notes": "Starter city for compact urban context.", "eea_station_selection_notes": "Use selected Amsterdam EEA station after file verification.", "wikipedia_url": "https://en.wikipedia.org/wiki/Amsterdam"},
    {"city_id": "warsaw_pl", "city_name": "Warsaw", "city_name_normalized": "warsaw", "country_code": "PL", "latitude": 52.2297, "longitude": 21.0122, "population": None, "area_km2": None, "population_density": None, "mapping_notes": "Starter city for Central and Eastern Europe.", "eea_station_selection_notes": "Use selected Warsaw EEA station after file verification.", "wikipedia_url": "https://en.wikipedia.org/wiki/Warsaw"},
    {"city_id": "prague_cz", "city_name": "Prague", "city_name_normalized": "prague", "country_code": "CZ", "latitude": 50.0755, "longitude": 14.4378, "population": None, "area_km2": None, "population_density": None, "mapping_notes": "Starter city for Central European coverage.", "eea_station_selection_notes": "Use selected Prague EEA station after file verification.", "wikipedia_url": "https://en.wikipedia.org/wiki/Prague"},
]

REQUIRED_COLUMNS = ["city_id", "city_name", "city_name_normalized", "country_code", "latitude", "longitude"]

def build_city_reference() -> pd.DataFrame:
    df = pd.DataFrame(CITY_RECORDS)
    expected_ids = df["city_name_normalized"] + "_" + df["country_code"].str.lower()
    if not (df["city_id"] == expected_ids).all():
        raise ValueError("city_id must follow <normalized_city>_<country_code>")
    if not df["city_id"].is_unique:
        raise ValueError("city_id values must be unique")
    if df[REQUIRED_COLUMNS].isna().any().any():
        raise ValueError("required city reference fields must not be null")
    if not df["latitude"].between(-90, 90).all() or not df["longitude"].between(-180, 180).all():
        raise ValueError("invalid coordinates")
    return df

def write_city_reference(output_dir: Path = Path("data/silver")) -> tuple[Path, Path]:
    output_dir.mkdir(parents=True, exist_ok=True)
    df = build_city_reference()
    csv_path = output_dir / "city_reference.csv"
    parquet_path = output_dir / "city_reference.parquet"
    df.to_csv(csv_path, index=False)
    df.to_parquet(parquet_path, index=False)
    return csv_path, parquet_path

city_reference = build_city_reference()
city_reference


## Validation / Quality Checks
Validate exact city count, unique IDs, non-null join keys, country-code format, and coordinate ranges.

In [ ]:
df = build_city_reference()
assert len(df) == 8
assert df["city_id"].is_unique
assert set(df["country_code"].str.len()) == {2}
assert df[REQUIRED_COLUMNS].notna().all().all()
print("city reference validation passed")


## Results
The notebook defines the stable city join backbone for all later notebooks.

## Limitations
City coordinates are fixed city-center approximations and not station-specific geometry.

## Next step
Run notebook `03` to process EEA historical batch files against `city_id`.